In [ ]:
import json, re
from collections import defaultdict

with open('admission_data_v2.json', encoding='utf-8') as f:
    data = json.load(f)

# ─────────────────────────────────────────────────────────────
# 전략:
# 1. 모집인원 → enriched_content의 [전형별 모집인원 요약] 파싱 → 학과별 1 row
# 2. 지원자격/제출서류/수능최저/평가방법/전형방법 →
#    (university, admission_type, track, section) 기준으로 인접 청크 병합
#    → 1 row per (uni, adm, track, section)
# 3. 기타/유의사항/전형일정 → 원본 유지 (낮은 QA 빈도)
# ─────────────────────────────────────────────────────────────

SECTION_TO_INFO = {
    '지원자격': '지원자격',
    '제출서류': '제출서류',
    '수능최저': '수능최저',
    '평가방법': '평가방법',
    '전형방법': '평가방법',
    '전형일정': '전형일정',
    '유의사항': '유의사항',
    '기타':     '기타',
}
MERGE_SECTIONS = set(SECTION_TO_INFO.keys())

new_rows = []
global_seq = defaultdict(int)  # per (university, admission_type)

def next_doc_id(university, admission_type):
    key = (university, admission_type)
    global_seq[key] += 1
    return f"{university}_{admission_type}_{global_seq[key]:04d}"


# ── STEP 1: 모집인원 (학과별 분리) ───────────────────────────
# enriched_content 안의 [전형별 모집인원 요약] 블록에서
# "대학교명 수시/정시 학과명 모집인원 총 N명" 패턴을 파싱해 1학과 = 1 row 생성

pattern = re.compile(r'(.+?)\s+(수시|정시)\s+(.+?)\s+모집인원\s+총\s+(\d+)명')
seen_dept_keys = set()

for doc in sorted(data, key=lambda d: d['doc_id']):
    ec = doc.get('enriched_content', '') or ''
    if '[전형별 모집인원 요약]' not in ec:
        continue

    summary_part = ec.split('[전형별 모집인원 요약]')[1].strip()
    lines = [l.strip() for l in summary_part.split('\n') if l.strip()]
    track = doc.get('track', '') or ''

    for line in lines:
        m = pattern.match(line)
        if not m:
            continue
        university, admission_type, major, count = \
            m.group(1), m.group(2), m.group(3), m.group(4)

        # (대학, 수시/정시, 전형, 학과) 조합 중복 방지
        key = (university, admission_type, track, major)
        if key in seen_dept_keys:
            continue
        seen_dept_keys.add(key)

        track_label = f" {track}전형" if track and track != '기타' else ""
        content = f"{university} {admission_type}{track_label} {major} 모집인원은 {count}명이다."

        new_rows.append({
            "doc_id":         next_doc_id(university, admission_type),
            "university":     university,
            "admission_type": admission_type,
            "track":          track if track and track != '기타' else None,
            "major":          major,
            "info_type":      "모집인원",
            "content":        content,
        })


# ── STEP 2: 나머지 섹션 (청크 병합) ──────────────────────────
# 같은 (대학, 수시/정시, 전형, 섹션)에 속한 청크를 page 순으로 합쳐 1 row 생성

groups = defaultdict(list)
for doc in sorted(data, key=lambda d: (
    d['university'], d['admission_type'], d.get('track', ''), d.get('page', 0)
)):
    section = doc.get('section', '')
    if section not in MERGE_SECTIONS:
        continue
    group_key = (
        doc['university'],
        doc['admission_type'],
        doc.get('track', '') or '',
        section,
    )
    groups[group_key].append(doc)

for (university, admission_type, track, section), docs in groups.items():
    # 중복 텍스트 제거 후 병합
    seen_texts = set()
    merged_parts = []
    for doc in docs:
        text = (doc.get('content') or '').strip()
        if text and text not in seen_texts:
            seen_texts.add(text)
            merged_parts.append(text)

    if not merged_parts:
        continue

    track_label = f"[{track}전형]" if track and track not in ('기타', '') else ""
    content = (
        f"[{university}] [{admission_type}]{track_label} [{section}]\n"
        + '\n'.join(merged_parts)
    )

    new_rows.append({
        "doc_id":         next_doc_id(university, admission_type),
        "university":     university,
        "admission_type": admission_type,
        "track":          track if track and track not in ('기타', '') else None,
        "major":          None,
        "info_type":      SECTION_TO_INFO.get(section, section),
        "content":        content,
    })


# ── 저장 ─────────────────────────────────────────────────────
with open('admission_data_v3.json', 'w', encoding='utf-8') as f:
    json.dump(new_rows, f, ensure_ascii=False, indent=2)

print(f"총 {len(new_rows)}개 rows 저장 완료")

# info_type 분포 확인
from collections import Counter
print(Counter(r['info_type'] for r in new_rows))